# Day 6 — End-to-End ML Pipeline & Business Insights


## 1. Imports and project paths

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
RAW = ROOT / "data" / "raw"
PROCESSED = ROOT / "data" / "processed"
OUT = ROOT / "outputs"
FINAL = OUT / "day6_business_insights"
FINAL.mkdir(parents=True, exist_ok=True)

def read_csv(path):
    return pd.read_csv(path) if path.exists() else None

def first_existing(paths):
    for path in paths:
        if path.exists():
            try:
                return path, pd.read_csv(path)
            except Exception:
                pass
    return None, None

## 2. Load raw datasets and prior-day outputs

In [2]:
customers = read_csv(RAW/"customers.csv")
restaurants = read_csv(RAW/"restaurants.csv")
orders = read_csv(RAW/"orders.csv")
deliveries = read_csv(RAW/"deliveries.csv")
reviews = read_csv(RAW/"reviews.csv")

customer_path, customer_segments = first_existing([
    PROCESSED/"customer_features_clustered.csv",
    OUT/"customer_segmentation"/"customer_features_clustered.csv",
    OUT/"customer_segmentation"/"customer_segments.csv"])
restaurant_path, restaurant_segments = first_existing([
    PROCESSED/"restaurant_features_clustered.csv",
    OUT/"restaurant_segmentation"/"restaurant_features_clustered.csv",
    OUT/"restaurant_segmentation"/"restaurant_segments.csv"])
delivery_metrics_path, delivery_metrics = first_existing([
    OUT/"delivery_time_prediction"/"model_comparison.csv",
    OUT/"delivery_time_prediction"/"delivery_model_comparison.csv"])
demand_metrics_path, demand_metrics = first_existing([OUT/"demand_forecasting"/"model_comparison.csv"])
forecast_path, demand_forecast = first_existing([OUT/"demand_forecasting"/"next_24_hour_demand_forecast.csv"])

for name, df in {"customers":customers,"restaurants":restaurants,"orders":orders,
                 "deliveries":deliveries,"reviews":reviews,
                 "customer segments":customer_segments,"restaurant segments":restaurant_segments,
                 "delivery metrics":delivery_metrics,"demand metrics":demand_metrics,
                 "demand forecast":demand_forecast}.items():
    print(name, ":", df.shape if df is not None else "not found")

customers : (300, 5)
restaurants : (45, 6)
orders : (1800, 8)
deliveries : (1800, 9)
reviews : (1147, 6)
customer segments : (300, 15)
restaurant segments : (45, 18)
delivery metrics : not found
demand metrics : (2, 4)
demand forecast : (24, 2)


## 3. Customer and restaurant segmentation

In [3]:
insights = []
def cluster_profile(frame, label_candidates, entity_name, output_name):
    if frame is None:
        print(f"{entity_name} segmentation output not found. Run its Day 2 notebook first.")
        return
    label = next((c for c in label_candidates if c in frame.columns), None)
    display(frame.head())
    if label is None:
        print(f"No recognized cluster column in {entity_name} output.")
        return
    profile = frame.groupby(label).size().rename("entity_count").reset_index()
    profile.to_csv(FINAL/output_name,index=False)
    display(profile)
    insights.append({"Area":entity_name,"Evidence":f"{len(profile)} clusters in saved output",
                     "Recommendation":"Profile cluster behavior and compare relevant business KPIs before choosing interventions."})

cluster_profile(customer_segments,["kmeans_cluster","customer_segment","cluster"],
                "Customer segmentation","customer_segment_profile.csv")
cluster_profile(restaurant_segments,["kmeans_cluster","restaurant_segment","cluster"],
                "Restaurant segmentation","restaurant_segment_profile.csv")

,customer_id,age,gender,city,total_orders,weekend_orders,late_night_orders,total_spending,average_order_value,average_rating_given,ordering_frequency_per_month,kmeans_cluster,dbscan_cluster,pca_1,pca_2
0,CUST0001,22,Male,Pune,8.0,2.0,2.0,3054.03,381.75,3.43,3.03,1,0,0.494955,-0.915690
1,CUST0002,55,Prefer not to say,Pune,6.0,1.0,1.0,3241.89,540.31,3.50,3.08,0,0,-0.227563,-0.104510
2,CUST0003,49,Female,Pune,4.0,3.0,0.0,2328.99,582.25,3.33,3.56,0,0,-0.617954,-0.037336
3,CUST0004,39,Prefer not to say,Hyderabad,1.0,1.0,0.0,750.85,750.85,3.00,30.44,2,-1,-3.131749,2.401382
4,CUST0005,38,Non-binary,Mumbai,2.0,1.0,0.0,1801.33,900.66,3.00,5.54,0,0,-1.452925,0.735679


,kmeans_cluster,entity_count
0,0,163
1,1,128
2,2,9


,restaurant_id,restaurant_name,category,city,restaurant_rating,avg_prep_time_minutes,total_orders,cancelled_orders,average_prep_time,revenue,average_delivery_time,customer_review_average,average_rating,cancellation_rate,kmeans_cluster,dbscan_cluster,pca_1,pca_2
0,REST001,Restaurant 001,Cafe,Mumbai,3.8,18,37,4,18.0,16384.27,46.34,4.000000,4.00,0.108,1,0,-1.663476,-0.549937
1,REST002,Restaurant 002,Chinese,Mumbai,4.0,14,44,6,14.0,20311.45,49.21,3.666667,3.67,0.136,1,0,-0.901545,0.739448
2,REST003,Restaurant 003,Cafe,Chennai,4.1,23,39,1,23.0,20143.21,53.54,3.517241,3.52,0.026,2,0,0.575597,-0.048388
3,REST004,Restaurant 004,Desserts,Bengaluru,3.7,17,46,4,17.0,23259.45,43.62,4.071429,4.07,0.087,1,0,-1.617475,1.597058
4,REST005,Restaurant 005,Healthy,Mumbai,4.3,32,48,1,32.0,25764.34,63.34,3.257143,3.26,0.021,2,0,2.854230,1.432152


,kmeans_cluster,entity_count
0,0,11
1,1,21
2,2,13


## 4. Customer sentiment summary

In [4]:
if reviews is None:
    print("reviews.csv not found.")
else:
    text_col = next((c for c in ["review_text","review","comment","feedback","text"] if c in reviews.columns),None)
    rating_col = next((c for c in ["rating","review_rating","customer_rating"] if c in reviews.columns),None)
    if text_col and rating_col:
        reviews[rating_col] = pd.to_numeric(reviews[rating_col],errors="coerce")
        reviews["sentiment_label"] = np.select(
            [reviews[rating_col]>=4,reviews[rating_col]==3,reviews[rating_col]<=2],
            ["positive","neutral","negative"],default="unknown")
        sentiment = reviews["sentiment_label"].value_counts().rename_axis("sentiment").reset_index(name="reviews")
        sentiment.to_csv(FINAL/"sentiment_summary.csv",index=False)
        display(sentiment)
        negative = reviews.loc[reviews["sentiment_label"]=="negative",text_col].fillna("").astype(str)
        terms = negative.str.lower().str.replace(r"[^a-z\s]"," ",regex=True).str.split().explode()
        stop = {"the","and","for","was","with","this","that","are","not","but","you","very","too","from","have","had","they","our","my","is","to","of","a","in","it","on"}
        terms = terms[~terms.isin(stop) & (terms.str.len()>2)]
        top_terms = terms.value_counts().head(15).rename_axis("term").reset_index(name="count")
        top_terms.to_csv(FINAL/"frequent_negative_review_terms.csv",index=False)
        display(top_terms)
        insights.append({"Area":"Customer feedback","Evidence":f"{int((reviews.sentiment_label=='negative').sum())} negative reviews by rating proxy",
                         "Recommendation":"Review examples from frequent complaint themes before deciding which service issue to fix first."})
    else:
        print("Review text/rating columns not recognized; inspect reviews.csv columns.")

,sentiment,reviews
0,positive,686
1,neutral,327
2,negative,134


,term,count
0,food,67
1,packaging,37
2,damaged,37
3,support,37
4,slow,37
5,delivery,35
6,late,35
7,cold,35
8,quality,32
9,disappointing,32


## 5. Delivery time and tip prediction

In [5]:
if deliveries is not None and "delivery_time_minutes" in deliveries.columns:
    delivery_data = deliveries.copy()
    if "delivery_status" in delivery_data.columns:
        delivery_data = delivery_data[delivery_data["delivery_status"].astype(str).str.lower()=="completed"]
    delivery_data["delivery_time_minutes"] = pd.to_numeric(delivery_data["delivery_time_minutes"],errors="coerce")
    delivery_data = delivery_data.dropna(subset=["delivery_time_minutes"])
    print("Completed deliveries with valid time:",len(delivery_data))
    display(delivery_data["delivery_time_minutes"].describe().to_frame())
else:
    print("Delivery-time target not found in expected deliveries.csv schema.")

if delivery_metrics is not None:
    print("Saved Day 4 model comparison:")
    display(delivery_metrics)
else:
    print("Day 4 model comparison CSV not found. Run day4_delivery_time_prediction.ipynb and save its metrics to outputs/delivery_time_prediction/model_comparison.csv.")

tip_col = next((c for c in ["tip_amount","tip","driver_tip"] if orders is not None and c in orders.columns),None)
if tip_col:
    orders[tip_col] = pd.to_numeric(orders[tip_col],errors="coerce")
    display(orders[tip_col].describe().to_frame())
else:
    print("No tip field found in orders.csv; a tip prediction model cannot be reported as completed.")

Completed deliveries with valid time: 1624


,delivery_time_minutes
count,1624.000000
mean,52.700123
std,15.859693
min,11.000000
25%,41.000000
50%,51.000000
75%,62.000000
max,125.000000


Day 4 model comparison CSV not found. Run day4_delivery_time_prediction.ipynb and save its metrics to outputs/delivery_time_prediction/model_comparison.csv.
No tip field found in orders.csv; a tip prediction model cannot be reported as completed.


## 6. Demand forecasting integration

In [6]:
if demand_metrics is not None:
    print("Day 5 model comparison:")
    display(demand_metrics)
else:
    print("Day 5 model comparison not found. Run day5_hourly_demand_forecasting.ipynb first.")

if demand_forecast is not None:
    display(demand_forecast.head(24))
    time_col = next((c for c in ["hour","ds","timestamp"] if c in demand_forecast.columns),None)
    value_col = next((c for c in ["forecast_orders","yhat","predicted_orders"] if c in demand_forecast.columns),None)
    if time_col and value_col:
        demand_forecast[time_col] = pd.to_datetime(demand_forecast[time_col],errors="coerce")
        demand_forecast[value_col] = pd.to_numeric(demand_forecast[value_col],errors="coerce")
        display(demand_forecast.nlargest(5,value_col))
        demand_forecast.to_csv(FINAL/"reviewed_demand_forecast.csv",index=False)
        insights.append({"Area":"Demand planning","Evidence":f"Forecast includes {len(demand_forecast)} future time periods",
                         "Recommendation":"Use forecast peaks to plan a pilot for rider capacity; compare forecast against actual demand before scaling."})
else:
    print("Next-24-hour forecast not found.")

Day 5 model comparison:


,Model,MAE,RMSE,MAPE_percent
0,"ARIMA(1,1,1)",0.751345,0.913677,32.930339
1,Prophet,0.753813,0.917298,37.339180


,hour,forecast_orders
0,2026-03-31 23:00:00,0.78
1,2026-04-01 00:00:00,0.79
2,2026-04-01 01:00:00,0.79
3,2026-04-01 02:00:00,0.79
4,2026-04-01 03:00:00,0.79
5,2026-04-01 04:00:00,0.79
6,2026-04-01 05:00:00,0.79
7,2026-04-01 06:00:00,0.79
8,2026-04-01 07:00:00,0.79
9,2026-04-01 08:00:00,0.79


,hour,forecast_orders
1,2026-04-01 00:00:00,0.79
2,2026-04-01 01:00:00,0.79
3,2026-04-01 02:00:00,0.79
4,2026-04-01 03:00:00,0.79
5,2026-04-01 04:00:00,0.79


## 7. Integrated component status

In [7]:
status = pd.DataFrame([
    {"Component":"Customer segmentation","Status":"Available" if customer_segments is not None else "Missing — run Day 2 customer notebook"},
    {"Component":"Restaurant segmentation","Status":"Available" if restaurant_segments is not None else "Missing — run Day 2 restaurant notebook"},
    {"Component":"Sentiment analysis","Status":"Source reviews available" if reviews is not None else "Missing reviews.csv"},
    {"Component":"Delivery-time model metrics","Status":"Available" if delivery_metrics is not None else "Missing model comparison CSV"},
    {"Component":"Demand forecasting metrics","Status":"Available" if demand_metrics is not None else "Missing — run Day 5 notebook"},
    {"Component":"Future demand forecast","Status":"Available" if demand_forecast is not None else "Missing — run Day 5 notebook"}
])
status.to_csv(FINAL/"integrated_component_status.csv",index=False)
display(status)

,Component,Status
0,Customer segmentation,Available
1,Restaurant segmentation,Available
2,Sentiment analysis,Source reviews available
3,Delivery-time model metrics,Missing model comparison CSV
4,Demand forecasting metrics,Available
5,Future demand forecast,Available


## 8. Business recommendations and model limitations

In [8]:
recommendations = pd.DataFrame([
    {"Area":"Customer value","Evidence needed":"Compare spend, order frequency and cluster sizes in customer profiles.",
     "Action":"Target retention or loyalty offers only to groups whose measured behavior supports them."},
    {"Area":"Restaurant operations","Evidence needed":"Compare rating, preparation time and cancellation rate by restaurant cluster.",
     "Action":"Investigate groups with weak service KPIs and confirm causes using restaurant-level records."},
    {"Area":"Customer experience","Evidence needed":"Review negative-review counts, frequent terms and sample comments.",
     "Action":"Prioritize recurring issues only after checking the original feedback context."},
    {"Area":"Delivery delays","Evidence needed":"Use Day 4 test metrics and feature importance, not training performance alone.",
     "Action":"Pilot delay-reduction actions and monitor errors on new deliveries."},
    {"Area":"Rider capacity","Evidence needed":"Use Day 5 forecast peaks and compare with actual demand.",
     "Action":"Consider extra capacity during predicted peak periods, then evaluate impact on delays and cost."}
])
recommendations.to_csv(FINAL/"business_recommendations.csv",index=False)
display(recommendations)

limitations = [
    "The project datasets are synthetic and are for learning/demo purposes.",
    "Forecast quality depends on the historical time span and data quality.",
    "Time-series data must be split chronologically to avoid leakage.",
    "Rating-derived sentiment labels are proxies, not human-annotated ground truth.",
    "Feature importance and correlations do not prove causation.",
    "Tip prediction is not complete unless a tip target and separately evaluated model are available."
]
(FINAL/"model_limitations.txt").write_text("\n".join("- "+x for x in limitations),encoding="utf-8")
print("Day 6 deliverables saved to:",FINAL)

,Area,Evidence needed,Action
0,Customer value,"Compare spend, order frequency and cluster siz...",Target retention or loyalty offers only to gro...
1,Restaurant operations,"Compare rating, preparation time and cancellat...",Investigate groups with weak service KPIs and ...
2,Customer experience,"Review negative-review counts, frequent terms ...",Prioritize recurring issues only after checkin...
3,Delivery delays,"Use Day 4 test metrics and feature importance,...",Pilot delay-reduction actions and monitor erro...
4,Rider capacity,Use Day 5 forecast peaks and compare with actu...,Consider extra capacity during predicted peak ...


Day 6 deliverables saved to: /home/nineleaps/Downloads/uber_eats-ML/outputs/day6_business_insights
